## ***We want to predict whether a customer will purchase a product. Without pipeline***

In [195]:
# Step 1: Import Libraries

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [196]:
# step 2: Import the data
df = pd.read_csv('Dataset.csv')

In [197]:
# Step 3: Explore the Dataset

In [198]:
df.shape

(10035, 6)

In [199]:
df.head()

,Age,Salary,Experience,Gender,City,Purchased
0,56.0,60794.0,33,Female,Hyderabad,1
1,46.0,99459.0,18,Female,Delhi,0
2,32.0,70192.0,20,Female,Bangalore,0
3,25.0,47703.0,33,Female,Mumbai,0
4,NaN,104193.0,16,Female,Mumbai,1


In [200]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10035 entries, 0 to 10034
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Age         9915 non-null   float64
 1   Salary      9937 non-null   float64
 2   Experience  10035 non-null  int64  
 3   Gender      9955 non-null   object 
 4   City        10035 non-null  object 
 5   Purchased   10035 non-null  int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 470.5+ KB


In [201]:
df.isnull().sum()

Age           120
Salary         98
Experience      0
Gender         80
City            0
Purchased       0
dtype: int64

In [202]:
df.duplicated().sum()

35

### ***Next Step (Step 4)***
- Before training a model, we'll clean the data without using Pipeline.
- We'll manually perform:

    - Remove duplicate rows
    - Handle missing values
    - Encode categorical features
    - Scale numerical features
    - Split the data
    - Train Logistic Regression
    - Save the model
    - Load the model
    - Predict on new data

In [203]:
# handle duplicate values
df = df.drop_duplicates()

In [204]:
df.shape

(10000, 6)

In [205]:
# Separate Features and Target
X = df.drop("Purchased", axis=1)
y = df["Purchased"]

In [206]:
X.head()

,Age,Salary,Experience,Gender,City
0,56.0,60794.0,33,Female,Hyderabad
1,46.0,99459.0,18,Female,Delhi
2,32.0,70192.0,20,Female,Bangalore
3,25.0,47703.0,33,Female,Mumbai
4,NaN,104193.0,16,Female,Mumbai


In [207]:
y.head()

0    1
1    0
2    0
3    0
4    1
Name: Purchased, dtype: int64

In [208]:
#Step 8: Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### ***Handle Missing Values (Manually)***
-  We have missing values in:
    - Age
    - Salary
    - Gender
- We'll use SimpleImputer.

In [209]:
from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy="mean")

In [210]:
num_imputer.fit(
    X_train[["Age", "Salary"]]
)

SimpleImputer()

In [211]:
X_train[["Age", "Salary"]] = num_imputer.transform(
    X_train[["Age", "Salary"]]
)

In [212]:
X_test[["Age", "Salary"]] = num_imputer.transform(
    X_test[["Age", "Salary"]]
)

### ***Categorical Column***
- Now for Gender.

In [213]:
cat_imputer = SimpleImputer(strategy="most_frequent")

In [214]:
cat_imputer.fit(
    X_train[["Gender"]]
)

SimpleImputer(strategy='most_frequent')

In [215]:
X_train[["Gender"]] = cat_imputer.transform(
    X_train[["Gender"]]
)

X_test[["Gender"]] = cat_imputer.transform(
    X_test[["Gender"]]
)

## **Stop Here and Observe**
- Look how many objects we have already created.
    - num_imputer
    - cat_imputer
- These are not temporary.
- We must keep them because future data must also use these exact learned values.
- If tomorrow a new customer comes, we cannot calculate a new mean or a new most frequent value—we must reuse these trained imputers.

## ***Encode Categorical Features (Without Pipeline)***
- Our categorical columns are:
    - Gender
    - City

In [216]:
from sklearn.preprocessing import OneHotEncoder

In [217]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    drop='first'
)

In [218]:
encoder.fit(
    X_train[["Gender", "City"]]
)

OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

In [219]:
encoded_train = encoder.transform(
    X_train[["Gender", "City"]]
)

In [220]:
encoded_train = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(
        ["Gender", "City"]
    ),
    index=X_train.index
)

In [221]:
X_train = X_train.drop(
    ["Gender", "City"],
    axis=1
)

In [222]:
X_train = pd.concat(
    [X_train, encoded_train],
    axis=1
)

In [223]:
X_train.head()

,Age,Salary,Experience,Gender_Male,City_Delhi,City_Hyderabad,City_Mumbai
9254,32.0,110502.0,12,0.0,1.0,0.0,0.0
1561,43.0,47195.0,7,0.0,0.0,0.0,0.0
1670,58.0,105848.0,23,0.0,0.0,1.0,0.0
6087,42.0,92392.0,34,0.0,0.0,0.0,1.0
6669,35.0,111280.0,19,1.0,1.0,0.0,0.0


In [224]:
encoded_test = encoder.transform(
    X_test[["Gender", "City"]]
)

encoded_test = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(),
    index=X_test.index
)

X_test = X_test.drop(
    ["Gender", "City"],
    axis=1
)

X_test = pd.concat(
    [X_test, encoded_test],
    axis=1
)

In [225]:
encoded_test.head()

,Gender_Male,City_Delhi,City_Hyderabad,City_Mumbai
6252,1.0,1.0,0.0,0.0
4684,1.0,0.0,0.0,1.0
1731,0.0,0.0,1.0,0.0
4742,0.0,0.0,1.0,0.0
4521,1.0,0.0,1.0,0.0


In [226]:
X_test.head()

,Age,Salary,Experience,Gender_Male,City_Delhi,City_Hyderabad,City_Mumbai
6252,24.0,58286.0,10,1.0,1.0,0.0,0.0
4684,42.0,88358.0,6,1.0,0.0,0.0,1.0
1731,49.0,99782.0,14,0.0,0.0,1.0,0.0
4742,33.0,93905.0,1,0.0,0.0,1.0,0.0
4521,29.0,97192.0,14,1.0,0.0,1.0,0.0


### ***Until now, we have created 3 trained objects:***
   - num_imputer
   - cat_imputer
   - encoder
- Now we'll create the 4th object.

## ***Next Step : Feature Scaling (Without Pipeline)***
- Why do we need Scaling?
- Our numerical columns are:

> Feature  -->	Range
> Age	-->  18–60
> Experience --> 	0–35
> Salary	--> 20,000–120,000

- Notice the difference.
    
    - Age          → 18 - 60
    
    - Experience   → 0 - 35
    
    - Salary       → 20,000 - 120,000

> Salary has much larger values.

> Many ML algorithms (including Logistic Regression) perform better when numerical features are on a similar scale.

>So we'll use StandardScaler.

In [227]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [228]:
num_cols = ["Age", "Salary", "Experience"]
scaler.fit(X_train[num_cols])

StandardScaler()

In [229]:
X_train[num_cols] = scaler.transform(
    X_train[num_cols]
)

In [230]:
X_test[num_cols] = scaler.transform(
    X_test[num_cols]
)

In [231]:
Currenet work flow 

Dataset
     ↓
Remove Duplicates
     ↓
Train-Test Split
     ↓
Numerical Imputer
     ↓
Categorical Imputer
     ↓
OneHotEncoder
     ↓
StandardScaler

SyntaxError: invalid character '↓' (U+2193) (2556192402.py, line 4)


#### Stop and Observe

Look at the objects we have created so far.

```python
num_imputer

cat_imputer

encoder

scaler
```

Each object has **learned something** from the training data.

| Object      | What it learned             |
| ----------- | --------------------------- |
| num_imputer | Mean values                 |
| cat_imputer | Most frequent category      |
| encoder     | Unique categories           |
| scaler      | Mean and standard deviation |

None of these are temporary.

They are all required later for prediction.

---

#### Current Workflow

```text
Dataset
     ↓
Remove Duplicates
     ↓
Train-Test Split
     ↓
Numerical Imputer
     ↓
Categorical Imputer
     ↓
OneHotEncoder
     ↓
StandardScaler
```

---

#### Think Like an ML Engineer

Imagine tomorrow you receive a new customer:

```text
Age = 32
Salary = 50000
Experience = 6
Gender = Male
City = Delhi
```

Before prediction, what must happen?

```text
New Customer
      ↓
num_imputer.transform()
      ↓
cat_imputer.transform()
      ↓
encoder.transform()
      ↓
scaler.transform()
      ↓
model.predict()
```

This is already becoming a long sequence, and **we haven't even trained the model yet**.

In the next step, we'll:

1. Train the `LogisticRegression` model.
2. Save **all five trained objects** (`num_imputer`, `cat_imputer`, `encoder`, `scaler`, and `model`).
3. Load them again.
4. Predict for a new customer.

That's where the inconvenience of working **without a Pipeline** becomes very clear.


# **Train Logistic Regression**

In [232]:
from sklearn.linear_model import LogisticRegression

In [233]:
model = LogisticRegression()

In [234]:
model.fit(X_train, y_train)

LogisticRegression()

In [235]:
predictions = model.predict(X_test)

In [240]:
from sklearn.metrics import accuracy_score

In [241]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy :", accuracy)

Accuracy : 0.487


## ***here our motive is just implement the flow without pipeline not increase accuracy***

In [242]:
print(f"Accuracy : {accuracy * 100:.2f}%")

Accuracy : 48.70%


In [244]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.46      0.32      0.38       978
           1       0.50      0.65      0.56      1022

    accuracy                           0.49      2000
   macro avg       0.48      0.48      0.47      2000
weighted avg       0.48      0.49      0.47      2000



In [243]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[312 666]
 [360 662]]


In [ ]:
Meaning:

312  → True Negatives
666 → False Positives
360  → False Negatives
662 → True Positives

# ***These are 5 different trained objects. Let's save them.***

In [236]:
import pickle
pickle.dump(
    num_imputer,
    open("num_imputer.pkl", "wb")
)

pickle.dump(
    cat_imputer,
    open("cat_imputer.pkl", "wb")
)

pickle.dump(
    encoder,
    open("encoder.pkl", "wb")
)

pickle.dump(
    scaler,
    open("scaler.pkl", "wb")
)
pickle.dump(
    model,
    open("model.pkl", "wb")
)

# ***Load Everything Again***

In [237]:
num_imputer = pickle.load(
    open("num_imputer.pkl", "rb")
)

cat_imputer = pickle.load(
    open("cat_imputer.pkl", "rb")
)

encoder = pickle.load(
    open("encoder.pkl", "rb")
)

scaler = pickle.load(
    open("scaler.pkl", "rb")
)

model = pickle.load(
    open("model.pkl", "rb")
)

## 🔴prediction using new data

In [256]:
import pandas as pd

new_customer = pd.DataFrame({
    "Age": [30],
    "Salary": [50000],
    "Experience": [5],
    "Gender": ["Male"],
    "City": ["Delhi"]
})

new_customer

,Age,Salary,Experience,Gender,City
0,30,50000,5,Male,Delhi


In [257]:
new_customer[["Age", "Salary"]] = num_imputer.transform(
    new_customer[["Age", "Salary"]]
)

In [258]:
new_customer[["Gender"]] = cat_imputer.transform(
    new_customer[["Gender"]]
)

In [259]:
encoded = encoder.transform(
    new_customer[["Gender", "City"]]
)

In [260]:
encoded = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(),
    index=new_customer.index
)

In [261]:
encoded

,Gender_Male,City_Delhi,City_Hyderabad,City_Mumbai
0,1.0,1.0,0.0,0.0


In [262]:
new_customer = new_customer.drop(
    ["Gender", "City"],
    axis=1
)

In [263]:
new_customer = pd.concat(
    [new_customer, encoded],
    axis=1
)

In [264]:
new_customer

,Age,Salary,Experience,Gender_Male,City_Delhi,City_Hyderabad,City_Mumbai
0,30.0,50000.0,5,1.0,1.0,0.0,0.0


In [265]:
num_cols = ["Age", "Salary", "Experience"]

new_customer[num_cols] = scaler.transform(
    new_customer[num_cols]
)

In [266]:
prediction = model.predict(new_customer)

print(prediction)

[1]


In [267]:
if prediction[0] == 1:
    print("Customer will Purchase")
else:
    print("Customer will Not Purchase")

Customer will Purchase


# ***Now observe the pain 😅***
- ***To predict just one customer, you had to remember this entire workflow:***

> **If you forget even a single preprocessing step—such as handling missing values, encoding categorical features, scaling numerical features, or maintaining the correct column order—the model may produce incorrect predictions or even raise an error.**

> This is exactly the problem that **Pipeline** and **ColumnTransformer** solve.

> Instead of manually remembering and applying every preprocessing step, you combine the entire workflow into a single object.

> Without Pipeline, the prediction workflow looks like this:

```text
New Customer
      │
      ▼
Numerical Imputer
      │
      ▼
Categorical Imputer
      │
      ▼
OneHotEncoder
      │
      ▼
Feature Scaling
      │
      ▼
Logistic Regression
      │
      ▼
Prediction
```

> With **Pipeline + ColumnTransformer**, all these steps are executed automatically.

- You simply write:

```python
pipeline.predict(new_customer)
```

- The pipeline automatically:

    1. Handles missing values
    2. Encodes categorical features
    3. Scales numerical features
    4. Maintains the correct feature order
    5. Sends the processed data to the trained model
    6. Returns the prediction

> **This is the biggest advantage of a Pipeline:** it makes the machine learning workflow **cleaner, reusable, less error-prone, and much easier to deploy.**
